In [1]:
!pip install kagglehub[pandas-datasets]

In [2]:
import kagglehub

# Baixa a versão mais recente do dataset de imagens
path = kagglehub.dataset_download("dataclusterlabs/fire-and-smoke-dataset")

print("O dataset foi baixado com sucesso em:", path)

# Listar o conteúdo baixado
import os
print("Arquivos e pastas no diretório do dataset:", os.listdir(path))

Using Colab cache for faster access to the 'fire-and-smoke-dataset' dataset.
O dataset foi baixado com sucesso em: /kaggle/input/fire-and-smoke-dataset
Arquivos e pastas no diretório do dataset: ['Annotations', 'Datacluster Fire and Smoke Sample']


In [3]:
images = os.path.join(path, 'Datacluster Fire and Smoke Sample', 'Datacluster Fire and Smoke Sample')
annotations = os.path.join(path, 'Annotations', 'Annotations')
print(os.listdir(images))
print(os.listdir(annotations))

['Datacluster Fire and Smoke Sample (15).jpg', 'Datacluster Fire and Smoke Sample (70).jpg', 'Datacluster Fire and Smoke Sample (22).jpg', 'Datacluster Fire and Smoke Sample (67).jpg', 'Datacluster Fire and Smoke Sample (66).jpg', 'Datacluster Fire and Smoke Sample (55).jpg', 'Datacluster Fire and Smoke Sample (46).jpg', 'Datacluster Fire and Smoke Sample (43).jpg', 'Datacluster Fire and Smoke Sample (84).jpg', 'Datacluster Fire and Smoke Sample (24).jpg', 'Datacluster Fire and Smoke Sample (21).jpg', 'Datacluster Fire and Smoke Sample (58).jpg', 'Datacluster Fire and Smoke Sample (86).jpg', 'Datacluster Fire and Smoke Sample (69).jpg', 'Datacluster Fire and Smoke Sample (75).jpg', 'Datacluster Fire and Smoke Sample (18).jpg', 'Datacluster Fire and Smoke Sample (12).jpg', 'Datacluster Fire and Smoke Sample (29).jpg', 'Datacluster Fire and Smoke Sample (32).jpg', 'Datacluster Fire and Smoke Sample (91).jpg', 'Datacluster Fire and Smoke Sample (26).jpg', 'Datacluster Fire and Smoke Sampl

In [4]:
base_dir = 'dataset'
for folder in ['train', 'val']:
  os.makedirs(os.path.join(base_dir, folder, 'images'), exist_ok=True)
  os.makedirs(os.path.join(base_dir, folder, 'labels'), exist_ok=True)
class_map = {"fire": 0, "smoke": 1}
images_list = []
annotations_list = []
for img in os.listdir(images):
  if img.endswith('.jpg'):
    images_list.append(img)
print(f'total de imagens: {len(images_list)}')
cut = int(0.8 * (len(images_list)))
train = images_list[:cut]
val = images_list[cut:]
print(f"""
imagens para treino:{len(train)}
imagens para validação:{len(val)}""")

total de imagens: 100

imagens para treino:80
imagens para validação:20


In [5]:
import shutil
for img in train:
  origem =os.path.join(images,img)
  destino = os.path.join(base_dir, 'train', 'images', img)
  shutil.copy(origem, destino)
for img in val:
  origem =os.path.join(images,img)
  destino = os.path.join(base_dir, 'val', 'images', img)
  shutil.copy(origem, destino)

In [6]:
import xml.etree.ElementTree as ET
# !cat '/root/.cache/kagglehub/datasets/dataclusterlabs/fire-and-smoke-dataset/versions/3/Annotations/Annotations/Datacluster Fire and Smoke Sample (12).xml'
xml = '/root/.cache/kagglehub/datasets/dataclusterlabs/fire-and-smoke-dataset/versions/3/Annotations/Annotations/Datacluster Fire and Smoke Sample (12).xml'
tree = ET.parse(xml)
root = tree.getroot()


obj = root.find('object')
name = obj.find('name').text


bndbox = obj.find('bndbox')
xmin = float(bndbox.find('xmin').text)
ymin = float(bndbox.find('ymin').text)
xmax = float(bndbox.find('xmax').text)
ymax = float(bndbox.find('ymax').text)

size = root.find('size')
w = float(size.find('width').text)
h = float(size.find('height').text)

class_id = class_map.get(name)
x_center = (xmin + xmax) / 2 / w
y_center = (ymin + ymax) / 2 / h
width = (xmax - xmin) / w
height = (ymax - ymin) / h

print(f"""{xmin} + {xmax} / 2 / {w} = {x_center} """)
# Annotation line format: <class_id> <x_center> <y_center> <width> <height> de https://labelformat.com/formats/object-detection/yolov11/
print(class_id, x_center, y_center, width, height)

1175.0 + 1256.25 / 2 / 3000.0 = 0.40520833333333334 
0 0.40520833333333334 0.60078125 0.027083333333333334 0.0765625


In [7]:
!pip install ultralytics
!pip install Ipython

In [8]:
from ultralytics import YOLO
output_labels_dir = "/kaggle/working/fire_and_smoke/labels"

os.makedirs(output_labels_dir, exist_ok=True)
model = YOLO('yolo11n.pt')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
